# EDA\n

Flinn BI technical challenge.\n\nThis notebook does lightweight profiling + relationship exploration across the provided CSVs (committed as dbt seeds). It also reproduces the **bonus insight activation funnel**.\n\nNotes:\n- The EDA is intentionally SQL-first using DuckDB so the same logic can be ported into dbt models.\n- Companion dbt-runnable queries live in `dbt/flinn_bi/analyses/`.\n

In [ ]:
from __future__ import annotations

from pathlib import Path

import duckdb


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "dbt" / "flinn_bi" / "dbt_project.yml").exists():
            return p
    raise FileNotFoundError("Could not locate repo root (missing dbt/flinn_bi/dbt_project.yml)")


REPO_ROOT = find_repo_root(Path.cwd())
SEEDS_DIR = REPO_ROOT / "dbt" / "flinn_bi" / "seeds"

SEED_FILES: dict[str, Path] = {
    "backend_events": SEEDS_DIR / "backend_events.csv",
    "hubspot_contacts": SEEDS_DIR / "hubspot_contacts.csv",
    "hubspot_deals": SEEDS_DIR / "hubspot_deals.csv",
    "hubspot_org": SEEDS_DIR / "hubspot_org.csv",
}

for name, path in SEED_FILES.items():
    assert path.exists(), f"Missing seed file: {name} at {path}"

con = duckdb.connect(database=":memory:")

try:
    import pandas as pd  # noqa: F401

    HAS_PANDAS = True
except Exception:
    HAS_PANDAS = False


def q(sql: str):
    cur = con.execute(sql)
    if HAS_PANDAS:
        return cur.fetchdf()
    return cur.fetchall()


# Create temp views so we can reference each dataset by name in SQL
for view_name, path in SEED_FILES.items():
    con.execute(
        "create or replace temp view "
        + view_name
        + " as select * from read_csv_auto('"
        + path.as_posix()
        + "', header=true);"
    )

q("show tables")


## 1) Quick profiling per file\n\nRow counts, candidate IDs, duplicates, and date ranges.

In [ ]:
q(
    """
with profiles as (
    select
        'backend_events' as table_name,
        count(*) as row_count,
        count(distinct event_id) as distinct_id_count,
        sum(case when event_id is null then 1 else 0 end) as id_nulls,
        (count(*) - count(distinct event_id)) as duplicate_id_rows,
        min(event_timestamp) as min_date,
        max(event_timestamp) as max_date
    from backend_events

    union all

    select
        'hubspot_contacts' as table_name,
        count(*) as row_count,
        count(distinct contact_id) as distinct_id_count,
        sum(case when contact_id is null then 1 else 0 end) as id_nulls,
        (count(*) - count(distinct contact_id)) as duplicate_id_rows,
        min(create_date) as min_date,
        max(create_date) as max_date
    from hubspot_contacts

    union all

    select
        'hubspot_org' as table_name,
        count(*) as row_count,
        count(distinct company_id) as distinct_id_count,
        sum(case when company_id is null then 1 else 0 end) as id_nulls,
        (count(*) - count(distinct company_id)) as duplicate_id_rows,
        min(create_date) as min_date,
        max(create_date) as max_date
    from hubspot_org

    union all

    select
        'hubspot_deals' as table_name,
        count(*) as row_count,
        count(distinct deal_id) as distinct_id_count,
        sum(case when deal_id is null then 1 else 0 end) as id_nulls,
        (count(*) - count(distinct deal_id)) as duplicate_id_rows,
        min(create_date) as min_date,
        max(create_date) as max_date
    from hubspot_deals
)
select * from profiles order by table_name;
"""
)


### Missingness checks (metric-critical fields)\n\nFocus on keys and join fields that will matter later (customers/ACV/retention).

In [ ]:
q(
    """
select
  'backend_events' as table_name,
  sum(case when event_timestamp is null then 1 else 0 end) as event_timestamp_nulls,
  sum(case when user_id is null then 1 else 0 end) as user_id_nulls,
  sum(case when organization_id is null then 1 else 0 end) as organization_id_nulls
from backend_events

union all

select
  'hubspot_contacts' as table_name,
  sum(case when create_date is null then 1 else 0 end) as event_timestamp_nulls,
  sum(case when email is null or trim(email) = '' then 1 else 0 end) as user_id_nulls,
  sum(case when hubspot_company_id is null then 1 else 0 end) as organization_id_nulls
from hubspot_contacts

union all

select
  'hubspot_deals' as table_name,
  sum(case when create_date is null then 1 else 0 end) as event_timestamp_nulls,
  sum(case when amount is null then 1 else 0 end) as user_id_nulls,
  sum(case when hubspot_company_id is null then 1 else 0 end) as organization_id_nulls
from hubspot_deals

order by table_name;
"""
)


## 2) Relationship exploration\n\nHow HubSpot objects relate, plus whether backend events can be linked to HubSpot entities.

In [ ]:
q(
    """
-- Contacts → orgs
select
  count(*) as contacts,
  count(distinct hubspot_company_id) as distinct_company_ids,
  sum(case when o.company_id is null then 1 else 0 end) as contacts_missing_company
from hubspot_contacts c
left join hubspot_org o
  on c.hubspot_company_id = o.company_id;
"""
)

q(
    """
-- Deals → orgs
select
  count(*) as deals,
  count(distinct hubspot_company_id) as distinct_company_ids,
  sum(case when o.company_id is null then 1 else 0 end) as deals_missing_company
from hubspot_deals d
left join hubspot_org o
  on d.hubspot_company_id = o.company_id;
"""
)


In [ ]:
q(
    """
-- Backend → HubSpot linking: user email only appears reliably on UserCreated events
select
  event_name,
  count(*) as n,
  sum(case when json_extract_string(event_properties, '$.user.email') is null then 1 else 0 end) as missing_user_email,
  round(
    100.0 * sum(case when json_extract_string(event_properties, '$.user.email') is null then 1 else 0 end) / count(*),
    1
  ) as pct_missing_user_email
from backend_events
group by 1
order by n desc
limit 12;
"""
)


In [ ]:
q(
    """
with user_created as (
  select
    user_id,
    organization_id,
    lower(json_extract_string(event_properties, '$.user.email')) as email,
    min(event_timestamp) as user_created_ts
  from backend_events
  where event_name = 'UserCreated'
  group by 1, 2, 3
), contact_email_map as (
  select
    contact_id,
    hubspot_company_id,
    lower(email) as email
  from hubspot_contacts
)
select
  count(distinct uc.user_id) as new_users,
  count(distinct case when cem.contact_id is not null then uc.user_id end) as new_users_matched_to_hubspot_contact,
  round(
    100.0 * count(distinct case when cem.contact_id is not null then uc.user_id end) / nullif(count(distinct uc.user_id), 0),
    1
  ) as pct_new_users_matched,
  count(distinct uc.organization_id) as backend_orgs,
  count(distinct cem.hubspot_company_id) as hubspot_companies_mapped
from user_created uc
left join contact_email_map cem
  on cem.email = uc.email;
"""
)


In [ ]:
q(
    """
-- Backend organization_id (UUID) can be mapped 1:1 to a HubSpot company_id via UserCreated.email → contacts.email
with user_created as (
  select
    organization_id as backend_org_id,
    lower(json_extract_string(event_properties, '$.user.email')) as email
  from backend_events
  where event_name = 'UserCreated'
), email_to_company as (
  select lower(email) as email, hubspot_company_id as company_id
  from hubspot_contacts
), backend_org_to_company as (
  select
    uc.backend_org_id,
    count(distinct etc.company_id) as mapped_company_ids
  from user_created uc
  left join email_to_company etc
    on etc.email = uc.email
  group by 1
)
select
  count(*) as backend_orgs,
  count(case when mapped_company_ids = 1 then 1 end) as orgs_map_to_one_company,
  count(case when mapped_company_ids > 1 then 1 end) as orgs_map_to_multiple_companies,
  count(case when mapped_company_ids = 0 then 1 end) as orgs_unmapped
from backend_org_to_company;
"""
)


## 3) Data quality findings\n

Impact on metrics.\n\nTop issues / quirks to keep in mind for Q1–Q3 modeling:\n- **Backend ↔ HubSpot overlap is partial:** backend events cover **37** orgs (≈ **14.6%** of HubSpot companies). Retention based on product events will be for that subset unless we define a broader “active” signal.\n- **User email is mostly absent on backend events:** for most event types, `event_properties.user.email` is missing; we should build a stable mapping table from `UserCreated` events and then join other backend events via `user_id` / `organization_id`.\n- **Event volume skew:** `TokenGenerated` is the most common event; if it’s automated, including it as “activity” could inflate retention.\n- **HubSpot relationships are clean:** contacts and deals both have non-null `hubspot_company_id` and match 100% to `hubspot_org` in this dataset.\n

## 4) Bonus insight\n

Activation funnel.\n\n**Question:** What share of new users run `SearchExecuted` within 7 days of `UserCreated`?\n\nDefinitions used here:\n- **New user:** distinct `user_id` with a `UserCreated` event, using the earliest timestamp per user.\n- **Activated:** has at least one `SearchExecuted` within `[user_created_ts, user_created_ts + 7 days]` (inclusive).\n

In [ ]:
q(
    """
with user_created as (
  select
    user_id,
    min(event_timestamp) as user_created_ts
  from backend_events
  where event_name = 'UserCreated'
    and user_id is not null
    and event_timestamp is not null
  group by 1
), activated as (
  select distinct uc.user_id
  from user_created uc
  join backend_events e
    on e.user_id = uc.user_id
   and e.event_name = 'SearchExecuted'
   and e.event_timestamp is not null
   and e.event_timestamp >= uc.user_created_ts
   and e.event_timestamp <= uc.user_created_ts + interval 7 day
)
select
  (select count(*) from user_created) as new_users,
  (select count(*) from activated) as activated_users,
  round(
    100.0 * (select count(*) from activated) / nullif((select count(*) from user_created), 0),
    1
  ) as activation_rate_pct;
"""
)
